# 🧠 Sprint 1: Train & Evaluate

---

## Model: YOLO11x

### Why YOLO11x?

| Model | Released | mAP COCO | Params | Decision |
|---|---|---|---|---|
| YOLOv8x | Jan 2023 | 53.9 | 68M | Older baseline |
| YOLOv9e | Feb 2024 | 55.6 | 58M | Good but superseded |
| YOLO11x | **Sep 2024** | **54.7** | **56M** | **✓ Selected** |

YOLO11 is the latest Ultralytics architecture. The `x` (extra-large) variant maximises detection accuracy at the cost of training time — appropriate for a clinical decision support tool where accuracy is critical and inference latency is not a primary constraint at this stage. At `imgsz=1280`, it fits comfortably in the 14GB VRAM of a Tesla T4 at `batch=4`.

### Why imgsz=1280?

Original GRAZPEDWRI-DX images are 2000–3000px. Wrist fractures, pronator signs and periosteal reactions are **subtle, small findings** — downsizing to 640px loses critical detail. 1280px retains enough resolution to detect fine-grained features while remaining trainable on a T4 GPU.

---

## Training Configuration Summary

| Parameter | Value | Rationale |
|---|---|---|
| epochs | 100 | Ceiling — early stopping (patience=25) typically triggers at 60–90 |
| imgsz | 1280 | Preserves fine-grained finding detail |
| batch | 4 | Safe for 14GB VRAM at imgsz=1280 |
| optimizer | AdamW | Better generalisation than SGD for medical imaging |
| lr0 | 0.001 | Standard starting LR for AdamW |
| cos_lr | True | Cosine decay — smooth LR reduction, avoids late-training instability |
| warmup_epochs | 5 | Gradual LR ramp-up prevents early gradient explosion |
| patience | 25 | Stop if val mAP doesn't improve for 25 consecutive epochs |
| mosaic | 1.0 | Mix 4 images — crucial for rare class exposure |
| mixup | 0.15 | Blend two images — improves generalisation |
| copy_paste | 0.3 | Copy rare findings into other images — extra rare class augmentation |
| flipud | 0.0 | **Disabled** — vertical flip is anatomically invalid for wrist X-rays |
| hsv_s, hsv_h | 0.0 | **Disabled** — X-rays are grayscale; saturation/hue irrelevant |
| cls | 1.5 | Upweight classification loss — penalises class confusion more heavily |
| box | 8.0 | Upweight box loss — prioritises accurate localisation |

---

## What Happens During Training

After every epoch, Ultralytics automatically:
- Runs the full val split through the model
- Computes `val/box_loss`, `val/cls_loss`, `val/dfl_loss`
- Computes `metrics/mAP50(B)` and `metrics/mAP50-95(B)`
- Saves `best.pt` whenever val mAP improves
- Saves `last.pt` after every epoch

All per-epoch metrics are written to `results.csv` inside the run folder.

---

## Restart Safety

> ✅ **After any kernel restart — run Cell 2 first.**
> It restores all path variables and the `BEST_WEIGHTS` path.
> If training has already completed, skip Cell 4 and go straight to Cell 5 (curves).

---

## Execution Order

```
Cell 1  →  Install ultralytics
Cell 2  →  Paths & config              ⚠️ Always run after restart
Cell 3  →  Verify dataset              (confirm split sizes and yaml content)
Cell 4  →  Train YOLO11x               ⏱️ ~6–10 hours; skips if best.pt exists
Cell 5  →  Plot training curves
Cell 6  →  Evaluate on val + test splits
Cell 7  →  Per-class results table
Cell 8  →  Evaluation figures
Cell 9  →  Model selection & results summary
Cell 10 →  Evidence checklist
```


---
## 1 — Install dependencies


In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
print("ultralytics ready.")

ultralytics ready.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
amazon-sagemaker-jupyter-ai-q-developer 1.2.9 requires numpy<=2.0.1, but you have numpy 2.4.3 which is incompatible.
amazon-sagemaker-sql-magic 0.1.4 requires numpy<2, but you have numpy 2.4.3 which is incompatible.
autogluon-common 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.3 which is incompatible.
autogluon-core 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.3 which is incompatible.
autogluon-features 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have 

---
## 2 — Paths & config ⚠️ Run this after every kernel restart


In [2]:
from pathlib import Path
import torch, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# ── Auto-detect EFS ───────────────────────────────────────────────────────────
_CANDIDATES = [
    Path("/home/sagemaker-user/user-default-efs"),
    Path("/home/sagemaker-user"),
    Path.home() / "user-default-efs",
    Path.home(),
]
EFS      = next((c for c in _CANDIDATES if (c/"IronGear").exists()), _CANDIDATES[0])
IRONGEAR = EFS / "IronGear"

# ── Shared data (built by 01_data_processing.ipynb) ──────────────────────────
DATA_DIR     = IRONGEAR / "data"
YOLO_DIR     = DATA_DIR / "yolo_dataset"
DATASET_YAML = YOLO_DIR / "dataset.yaml"

# ── Sprint 1 outputs ──────────────────────────────────────────────────────────
SPRINT_DIR   = IRONGEAR / "Sprint1-POC"
MODELS_DIR   = SPRINT_DIR / "models"
REPORTS_DIR  = SPRINT_DIR / "reports"
FIG_DIR      = SPRINT_DIR / "figures"

for d in [MODELS_DIR, REPORTS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Model config ──────────────────────────────────────────────────────────────
MODEL_WEIGHTS = "yolo11x.pt"
RUN_NAME      = "sprint1_yolo11x"
BEST_WEIGHTS  = MODELS_DIR / RUN_NAME / "weights" / "best.pt"
LAST_WEIGHTS  = MODELS_DIR / RUN_NAME / "weights" / "last.pt"

# ── Training config ───────────────────────────────────────────────────────────
TRAIN_CFG = {
    # Core
    "epochs"       : 50,
    "imgsz"        : 640,
    "batch"        : 2,
    "patience"     : 10,       # early stopping
    "device"       : "0",      # GPU 0
    "workers"      : 4,
    "seed"         : 42,
    "cache"        : "disk",
    "exist_ok" : True,
    # Optimiser
    "optimizer"    : "AdamW",
    "lr0"          : 0.001,
    "lrf"          : 0.01,
    "cos_lr"       : True,
    "weight_decay" : 0.0005,
    "warmup_epochs": 5,
    # Augmentation
    "mosaic"       : 1.0,
    "mixup"        : 0.15,
    "copy_paste"   : 0.3,
    "degrees"      : 8,
    "scale"        : 0.8,
    "fliplr"       : 0.5,
    "flipud"       : 0.0,      # anatomically wrong
    "hsv_s"        : 0.0,      # grayscale X-rays
    "hsv_h"        : 0.0,
    # Loss
    "cls"          : 1.5,
    "box"          : 8.0,
    # Output
    "exist_ok"     : True,
    "verbose"      : True,
    "plots"        : True,
}

PROJECT_CLASSES = {
    0: "fracture",
    1: "metal_implant",
    2: "periosteal_reaction",
    3: "pronator_sign",
    4: "soft_tissue",
}

# ── Check prerequisites ───────────────────────────────────────────────────────
assert DATASET_YAML.exists(), (
    f"dataset.yaml not found. Run 01_data_processing.ipynb first.\n"
    f"Expected: {DATASET_YAML}"
)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram     = torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0

print("Config loaded ✓")
print(f"  GPU          : {gpu_name}")
print(f"  VRAM         : {vram:.1f} GB")
print(f"  Model        : {MODEL_WEIGHTS}")
print(f"  Dataset yaml : {DATASET_YAML}")
print(f"  Output dir   : {SPRINT_DIR}")
print(f"  Best weights : {BEST_WEIGHTS} {'✓ EXISTS' if BEST_WEIGHTS.exists() else '(not yet trained)'}")

Config loaded ✓
  GPU          : Tesla T4
  VRAM         : 15.6 GB
  Model        : yolo11x.pt
  Dataset yaml : /home/sagemaker-user/user-default-efs/IronGear/data/yolo_dataset/dataset.yaml
  Output dir   : /home/sagemaker-user/user-default-efs/IronGear/Sprint1-POC
  Best weights : /home/sagemaker-user/user-default-efs/IronGear/Sprint1-POC/models/sprint1_yolo11x/weights/best.pt ✓ EXISTS


---
## 3 — Verify dataset


In [3]:
print("Dataset verification:")
print(f"{'Split':<8} {'Images':>8} {'Labels':>8} {'Positive':>10} {'Negative':>10}")
print("-" * 48)
for split in ["train","val","test"]:
    n_img = len(list((YOLO_DIR/"images"/split).glob("*")))
    n_lbl = len(list((YOLO_DIR/"labels"/split).glob("*.txt")))
    n_pos = sum(1 for p in (YOLO_DIR/"labels"/split).glob("*.txt")
                if p.stat().st_size > 0)
    print(f"{split:<8} {n_img:>8,} {n_lbl:>8,} {n_pos:>10,} {n_lbl-n_pos:>10,}")
print()
print("dataset.yaml contents:")
print(DATASET_YAML.read_text())

Dataset verification:
Split      Images   Labels   Positive   Negative
------------------------------------------------
train      39,106   23,454     18,897      4,557
val         2,969    2,969      2,011        958
test        3,115    3,115      2,135        980

dataset.yaml contents:
# Iron Gear — GRAZPEDWRI-DX 5-class detection dataset
# Auto-generated by IronGear/01_data_processing.ipynb
# Reused by Sprint 1, Sprint 2, and Sprint 3

path: /home/sagemaker-user/user-default-efs/IronGear/data/yolo_dataset   # absolute path to dataset root
train: images/train
val:   images/val
test:  images/test

nc: 5   # number of classes

# Class names — index position = class ID used in label files
names:
  0: fracture
  1: metal_implant
  2: periosteal_reaction
  3: pronator_sign
  4: soft_tissue



---
## 4 — Train YOLO11x
> **Expected time on Tesla T4:** ~6–10 hours for 100 epochs at imgsz=1280.
> Early stopping (patience=25) typically stops around epoch 60–90.
> 
> After every epoch you will see validation metrics printed:
> - `val/box_loss`, `val/cls_loss` — decreasing = model improving
> - `metrics/mAP50(B)` — main accuracy metric, should increase over epochs
> - `best.pt` saved automatically whenever val mAP improves
> 
> ⚠️ **Do not interrupt this cell.** You can close the browser tab — SageMaker keeps running.


In [3]:
import subprocess
result = subprocess.run(
    ["nvidia-smi", "--query-compute-apps=pid,used_memory",
     "--format=csv,noheader"],
    capture_output=True, text=True
)
print("Processes using GPU:")
print(result.stdout if result.stdout.strip() else "None — GPU is clear ✓")

import torch
free = torch.cuda.mem_get_info()[0] / 1e9
total = torch.cuda.mem_get_info()[1] / 1e9
print(f"\nGPU memory: {free:.2f} GB free / {total:.2f} GB total")
print("✓ Safe to train" if free > 10 else "✗ Not enough free memory — shut down other kernels")

Processes using GPU:
None — GPU is clear ✓

GPU memory: 15.53 GB free / 15.64 GB total
✓ Safe to train


In [4]:
import shutil
old_run = MODELS_DIR / RUN_NAME
if old_run.exists():
    shutil.rmtree(old_run)
    print(f"✓ Deleted old run: {old_run}")
else:
    print("No old run found — clean start.")

✓ Deleted old run: /home/sagemaker-user/user-default-efs/IronGear/Sprint1-POC/models/sprint1_yolo11x


In [ ]:
import time

if BEST_WEIGHTS.exists():
    print(f"Training already completed. best.pt found at:")
    print(f"  {BEST_WEIGHTS}")
    print(f"\nSkipping training. Jump to Cell 5 (Training Curves).")
    print(f"To retrain: delete {MODELS_DIR/RUN_NAME} and re-run this cell.")
else:
    print(f"Starting YOLO11x training...")
    print(f"Config: epochs={TRAIN_CFG['epochs']}, imgsz={TRAIN_CFG['imgsz']}, "
          f"batch={TRAIN_CFG['batch']}\n")

    model   = YOLO(MODEL_WEIGHTS)
    t0      = time.time()

    results = model.train(
        data    = str(DATASET_YAML),
        project = str(MODELS_DIR),
        name    = RUN_NAME,
        **TRAIN_CFG,
    )

    elapsed = time.time() - t0
    print(f"\nTraining complete in {elapsed/3600:.2f} hours")
    print(f"best.pt  : {BEST_WEIGHTS} ({'✓' if BEST_WEIGHTS.exists() else '✗'})")
    print(f"last.pt  : {LAST_WEIGHTS} ({'✓' if LAST_WEIGHTS.exists() else '✗'})")

Starting YOLO11x training...
Config: epochs=50, imgsz=640, batch=2

Ultralytics 8.4.25 🚀 Python-3.12.9 torch-2.6.0 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=8.0, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=1.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/sagemaker-user/user-default-efs/IronGear/data/yolo_dataset/dataset.yaml, degrees=8, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sprint1_yolo11x

In [5]:
last_pt    = MODELS_DIR / RUN_NAME / "weights" / "last.pt"
best_pt    = MODELS_DIR / RUN_NAME / "weights" / "best.pt"
result_csv = MODELS_DIR / RUN_NAME / "results.csv"

if result_csv.exists():
    df         = pd.read_csv(result_csv)
    df.columns = df.columns.str.strip()
    completed  = len(df)
    best_mAP   = df["metrics/mAP50(B)"].max() if "metrics/mAP50(B)" in df.columns else 0
    print(f"Previous progress:")
    print(f"  Epochs completed : {completed} / {TRAIN_CFG['epochs']}")
    print(f"  Best mAP@0.5     : {best_mAP:.4f}")
    print(f"  Epochs remaining : {TRAIN_CFG['epochs'] - completed}")
else:
    completed = 0
    print("No previous run found — starting fresh")

Previous progress:
  Epochs completed : 50 / 50
  Best mAP@0.5     : 0.6694
  Epochs remaining : 0


---
## 5 — Training curves


In [6]:
results_csv = MODELS_DIR / RUN_NAME / "results.csv"
assert results_csv.exists(), f"results.csv not found — has training completed? {results_csv}"

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

n_epochs = len(df)
print(f"Training completed: {n_epochs} epochs")

# Print final epoch metrics
last = df.iloc[-1]
best_epoch = df["metrics/mAP50(B)"].idxmax() if "metrics/mAP50(B)" in df.columns else -1
best = df.iloc[best_epoch] if best_epoch >= 0 else last
print(f"\nBest epoch: {best_epoch+1}/{n_epochs}")
for col in ["metrics/mAP50(B)","metrics/mAP50-95(B)",
            "metrics/precision(B)","metrics/recall(B)"]:
    if col in df.columns:
        print(f"  {col:<35}: {best[col]:.4f}")

# Plot
plot_cols = [
    ("train/box_loss",      "Train Box Loss",  "#e74c3c"),
    ("train/cls_loss",      "Train Cls Loss",  "#3498db"),
    ("val/box_loss",        "Val Box Loss",    "#e67e22"),
    ("val/cls_loss",        "Val Cls Loss",    "#9b59b6"),
    ("metrics/mAP50(B)",    "mAP@0.5",         "#2ecc71"),
    ("metrics/mAP50-95(B)", "mAP@0.5:0.95",    "#1abc9c"),
]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(f"Training Curves — {RUN_NAME} ({n_epochs} epochs)",
             fontsize=13, fontweight="bold")

for ax, (col, title, color) in zip(axes.flatten(), plot_cols):
    if col in df.columns:
        ax.plot(df.index+1, df[col].values, color=color, linewidth=1.8)
        if "mAP" in col and best_epoch >= 0:
            ax.axvline(best_epoch+1, color="red", linestyle="--", alpha=0.6,
                       label=f"Best epoch {best_epoch+1}")
            ax.legend(fontsize=8)
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.grid(True, alpha=0.3)
        ax.set_facecolor("#f9f9f9")
    else:
        ax.set_visible(False)

plt.tight_layout()
p = FIG_DIR / "01_training_curves.png"
plt.savefig(p, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {p}")

Training completed: 50 epochs

Best epoch: 50/50
  metrics/mAP50(B)                   : 0.6694
  metrics/mAP50-95(B)                : 0.3797
  metrics/precision(B)               : 0.7037
  metrics/recall(B)                  : 0.6520

Saved: /home/sagemaker-user/user-default-efs/IronGear/Sprint1-POC/figures/01_training_curves.png


---
## 6 — Evaluate on val + test splits


In [7]:
assert BEST_WEIGHTS.exists(), f"best.pt not found: {BEST_WEIGHTS}"

# Always rewrite dataset.yaml with direct path (avoids symlink issues after restart)
yaml_text = f"""path: {YOLO_DIR}
train: images/train
val:   images/val
test:  images/test

nc: 5
names:
  0: fracture
  1: metal_implant
  2: periosteal_reaction
  3: pronator_sign
  4: soft_tissue
"""
DATASET_YAML.write_text(yaml_text)

eval_device = "0" if torch.cuda.is_available() else "cpu"
best_model  = YOLO(str(BEST_WEIGHTS))
all_metrics = {}

for split in ["val", "test"]:
    print(f"\n{'='*55}")
    print(f" Evaluating on '{split}' split")
    print(f"{'='*55}")

    ev = best_model.val(
        data    = str(DATASET_YAML),
        split   = split,
        conf    = 0.001,
        iou     = 0.6,
        device  = eval_device,
        plots   = True,
        verbose = True,
    )

    # Overall metrics
    try:
        rd = ev.results_dict if hasattr(ev, "results_dict") else {}
        metrics = {k: float(v) for k,v in rd.items() if isinstance(v,(int,float))}
    except Exception:
        metrics = {}

    # Per-class AP
    try:
        per_class = {}
        for i, cls_idx in enumerate(ev.ap_class_index.tolist()):
            name = PROJECT_CLASSES.get(cls_idx, str(cls_idx))
            per_class[name] = {
                "ap50"   : round(float(ev.box.ap50[i]), 4),
                "ap50_95": round(float(ev.box.ap[i]),   4),
            }
        metrics["per_class"] = per_class
    except Exception as e:
        print(f"Per-class extraction note: {e}")

    all_metrics[split] = metrics

    print(f"\n  {'Metric':<35} Value")
    print(f"  {'-'*45}")
    for k in ["metrics/mAP50(B)","metrics/mAP50-95(B)",
              "metrics/precision(B)","metrics/recall(B)"]:
        print(f"  {k:<35} {metrics.get(k,0):.4f}")

# Save
with open(REPORTS_DIR/"eval_metrics.json","w") as f:
    json.dump(all_metrics, f, indent=2)
print("\nMetrics saved to reports/eval_metrics.json")


 Evaluating on 'val' split
Ultralytics 8.4.26 🚀 Python-3.12.9 torch-2.6.0 CUDA:0 (Tesla T4, 14918MiB)
YOLO11x summary (fused): 191 layers, 56,832,799 parameters, 0 gradients, 194.4 GFLOPs
val: Fast image access ✅ (ping: 1.0±0.0 ms, read: 67.7±22.1 MB/s, size: 797.8 KB)
val: Scanning /mnt/custom-file-systems/efs/fs-06283b063577615fd_fsap-0dea7c00663be5b80/IronGear/data/yolo_dataset/labels/val.cache... 2969 images, 958 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2969/2969 518.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 186/186 1.7it/s 1:470.4ss
                   all       2969       3531        0.7      0.653      0.676       0.38
              fracture       1978       2675      0.942      0.861      0.947      0.563
         metal_implant        106        132      0.882      0.811      0.858      0.622
   periosteal_reaction        342        571      0.804       0.52      0.679      0.291
         pronator_si

---
## 7 — Per-class results table


In [8]:
test_m    = all_metrics.get("test", {})
per_class = test_m.get("per_class", {})

print("Per-class results on TEST set:")
print(f"{'Class':<25} {'AP@0.5':>8} {'AP@0.5:0.95':>13} {'Status'}")
print("-" * 60)
for cls in PROJECT_CLASSES.values():
    ap50   = per_class.get(cls, {}).get("ap50",   0)
    ap5095 = per_class.get(cls, {}).get("ap50_95", 0)
    status = "✓ Good" if ap50 >= 0.6 else "△ Moderate" if ap50 >= 0.35 else "✗ Weak"
    print(f"{cls:<25} {ap50:>8.4f} {ap5095:>13.4f}  {status}")
print("-" * 60)
print(f"{'Overall mAP@0.5':<25} {test_m.get('metrics/mAP50(B)',0):>8.4f} "
      f"{test_m.get('metrics/mAP50-95(B)',0):>13.4f}")
print()
print(f"Precision : {test_m.get('metrics/precision(B)',0):.4f}")
print(f"Recall    : {test_m.get('metrics/recall(B)',0):.4f}")

Per-class results on TEST set:
Class                       AP@0.5   AP@0.5:0.95 Status
------------------------------------------------------------
fracture                    0.9385        0.5495  ✓ Good
metal_implant               0.8646        0.6641  ✓ Good
periosteal_reaction         0.6494        0.2875  ✓ Good
pronator_sign               0.6654        0.3237  ✓ Good
soft_tissue                 0.3068        0.1317  ✗ Weak
------------------------------------------------------------
Overall mAP@0.5             0.6850        0.3913

Precision : 0.6932
Recall    : 0.6971


---
## 8 — Evaluation figures


In [9]:
val_m  = all_metrics.get("val",  {})
test_m = all_metrics.get("test", {})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Sprint 1 — YOLO11x Evaluation Results",
             fontsize=14, fontweight="bold")

COLORS_CLS = ["#e74c3c","#3498db","#2ecc71","#f39c12","#9b59b6"]
metric_keys = [
    ("metrics/mAP50(B)",     "mAP@0.5"),
    ("metrics/mAP50-95(B)",  "mAP@0.5:0.95"),
    ("metrics/precision(B)", "Precision"),
    ("metrics/recall(B)",    "Recall"),
]
labels    = [m[1] for m in metric_keys]
val_vals  = [val_m.get(m[0],0)  for m in metric_keys]
test_vals = [test_m.get(m[0],0) for m in metric_keys]

# Overall
x = np.arange(len(labels)); w = 0.35
b1 = axes[0].bar(x-w/2, val_vals,  w, label="Validation", color="#3498db", alpha=0.85)
b2 = axes[0].bar(x+w/2, test_vals, w, label="Test",       color="#e67e22", alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, fontsize=10)
axes[0].set_ylim(0, 1.15); axes[0].set_ylabel("Score")
axes[0].set_title("Overall Metrics", fontweight="bold")
axes[0].legend(fontsize=10); axes[0].grid(axis="y", alpha=0.3)
axes[0].axhline(0.5, color="gray", linestyle="--", alpha=0.4)
for bar in b1.patches + b2.patches:
    v = bar.get_height()
    if v > 0.02:
        axes[0].text(bar.get_x()+bar.get_width()/2, v+0.015,
                     f"{v:.3f}", ha="center", fontsize=8)

# Per-class AP@0.5
per = test_m.get("per_class", {})
cls_names = list(PROJECT_CLASSES.values())
ap50_vals = [per.get(c,{}).get("ap50",0) for c in cls_names]
bars = axes[1].bar(cls_names, ap50_vals, color=COLORS_CLS, edgecolor="white", width=0.6)
axes[1].set_ylim(0, 1.15); axes[1].set_ylabel("AP@0.5")
axes[1].set_title("Per-Class AP@0.5 — Test Set", fontweight="bold")
axes[1].tick_params(axis="x", rotation=18)
axes[1].grid(axis="y", alpha=0.3)
axes[1].axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="0.5 baseline")
axes[1].axhline(0.7, color="green", linestyle=":", alpha=0.5, label="0.7 target")
axes[1].legend(fontsize=8)
for bar, v in zip(bars, ap50_vals):
    if v > 0.01:
        axes[1].text(bar.get_x()+bar.get_width()/2, v+0.015,
                     f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
p = FIG_DIR / "02_evaluation_results.png"
plt.savefig(p, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {p}")

Saved: /home/sagemaker-user/user-default-efs/IronGear/Sprint1-POC/figures/02_evaluation_results.png


---
## 9 — Model selection & results summary


In [10]:
import datetime

test_m    = all_metrics.get("test", {})
per_class = test_m.get("per_class", {})

mAP50   = test_m.get("metrics/mAP50(B)",    0)
mAP5095 = test_m.get("metrics/mAP50-95(B)", 0)
prec    = test_m.get("metrics/precision(B)",0)
recall  = test_m.get("metrics/recall(B)",   0)

# Load training curve summary
results_csv = MODELS_DIR / RUN_NAME / "results.csv"
if results_csv.exists():
    df_r = pd.read_csv(results_csv)
    df_r.columns = df_r.columns.str.strip()
    n_epochs_trained = len(df_r)
    best_epoch_idx   = df_r["metrics/mAP50(B)"].idxmax() if "metrics/mAP50(B)" in df_r.columns else "N/A"
    best_val_mAP     = df_r["metrics/mAP50(B)"].max() if "metrics/mAP50(B)" in df_r.columns else 0
else:
    n_epochs_trained = "N/A"
    best_epoch_idx   = "N/A"
    best_val_mAP     = 0

summary_md = f"""# Iron Gear — Sprint 1 Results Summary
Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}

## Selected Model
**YOLO11x** — best.pt saved at:
`{BEST_WEIGHTS}`

## Training Config
| Parameter | Value |
|---|---|
| Epochs trained | {n_epochs_trained} |
| Best epoch | {best_epoch_idx} |
| Best val mAP@0.5 | {best_val_mAP:.4f} |
| Image size | {TRAIN_CFG['imgsz']} |
| Batch size | {TRAIN_CFG['batch']} |
| Optimizer | {TRAIN_CFG['optimizer']} |
| LR | {TRAIN_CFG['lr0']} cosine |
| Augmentation | mosaic, mixup={TRAIN_CFG['mixup']}, copy_paste={TRAIN_CFG['copy_paste']} |

## Test Set Metrics
| Metric | Score |
|---|---|
| mAP@0.5 | {mAP50:.4f} |
| mAP@0.5:0.95 | {mAP5095:.4f} |
| Precision | {prec:.4f} |
| Recall | {recall:.4f} |

## Per-Class AP@0.5 (Test)
| Class | AP@0.5 | AP@0.5:0.95 |
|---|---|---|
{''.join([f"| {c} | {per_class.get(c,{}).get('ap50',0):.4f} | {per_class.get(c,{}).get('ap50_95',0):.4f} |{chr(10)}" for c in PROJECT_CLASSES.values()])}
## Sprint 2 Plan
- Upgrade to 8-class detection (add bone_anomaly, bone_lesion, foreign_body)
- Add plaster cast binary classifier
- ClearML experiment tracking (MLOps Level 1)
- Automated retraining pipeline
- TTA (Test Time Augmentation) during evaluation
"""

(REPORTS_DIR/"sprint1_results_summary.md").write_text(summary_md)
print(summary_md)

# Iron Gear — Sprint 1 Results Summary
Generated: 2026-03-24 22:08

## Selected Model
**YOLO11x** — best.pt saved at:
`/home/sagemaker-user/user-default-efs/IronGear/Sprint1-POC/models/sprint1_yolo11x/weights/best.pt`

## Training Config
| Parameter | Value |
|---|---|
| Epochs trained | 50 |
| Best epoch | 49 |
| Best val mAP@0.5 | 0.6694 |
| Image size | 640 |
| Batch size | 2 |
| Optimizer | AdamW |
| LR | 0.001 cosine |
| Augmentation | mosaic, mixup=0.15, copy_paste=0.3 |

## Test Set Metrics
| Metric | Score |
|---|---|
| mAP@0.5 | 0.6850 |
| mAP@0.5:0.95 | 0.3913 |
| Precision | 0.6932 |
| Recall | 0.6971 |

## Per-Class AP@0.5 (Test)
| Class | AP@0.5 | AP@0.5:0.95 |
|---|---|---|
| fracture | 0.9385 | 0.5495 |
| metal_implant | 0.8646 | 0.6641 |
| periosteal_reaction | 0.6494 | 0.2875 |
| pronator_sign | 0.6654 | 0.3237 |
| soft_tissue | 0.3068 | 0.1317 |

## Sprint 2 Plan
- Upgrade to 8-class detection (add bone_anomaly, bone_lesion, foreign_body)
- Add plaster cast binary cla

---
## 10 — Evidence checklist


In [13]:
checks = {
    "Best model weights"   : BEST_WEIGHTS,
    "Training results CSV" : MODELS_DIR/RUN_NAME/"results.csv",
    "Training curves fig"  : FIG_DIR/"01_training_curves.png",
    "Evaluation results fig": FIG_DIR/"02_evaluation_results.png",
    "Eval metrics JSON"    : REPORTS_DIR/"eval_metrics.json",
    "Results summary MD"   : REPORTS_DIR/"sprint1_results_summary.md",
}
print("Notebook — Evidence Checklist")
print("=" * 45)
all_ok = True
for label, path in checks.items():
    ok = Path(str(path)).exists()
    all_ok = all_ok and ok
    print(f"  {'✓' if ok else '✗'} {label}")
print()
print("All outputs ready ✓" if all_ok else "Some items missing — check ✗ above.")
print("\nNext → Run demo.ipynb")

Notebook — Evidence Checklist
  ✓ Best model weights
  ✓ Training results CSV
  ✓ Training curves fig
  ✓ Evaluation results fig
  ✓ Eval metrics JSON
  ✓ Results summary MD

All outputs ready ✓

Next → Run demo.ipynb
